# MTG-BDH scaling grid — free Colab runner

Runs the L-shaped scaling grid on a Colab GPU, resuming across sessions.

**Before you start**, put the processed dataset in Drive (~74MB, one time):

```
MyDrive/mtg-bdh/data/FIN.PremierDraft/
    picks.npz  card_features.npz  vocab.json  ingest_stats.json
```

The raw 9GB export is *not* needed here — ingest it once locally and upload
the processed directory.

**Runtime → Change runtime type → T4 GPU** before running anything.

## 1. Check the accelerator

In [ ]:
!nvidia-smi -L || echo "NO GPU — set Runtime > Change runtime type > T4 GPU"

## 2. Install JAX with CUDA

Colab ships a JAX build that is often CPU-only or mismatched to the driver.
The assert is the point: without it a missing GPU silently costs ~100x, and
the grid looks merely slow rather than misconfigured.

In [ ]:
!pip install -q -U "jax[cuda12]"

import jax
print("jax", jax.__version__, "|", jax.default_backend(), "|", jax.devices())
assert jax.default_backend() == "gpu", "not on GPU — fix before running the grid"

## 3. Mount Drive and get the code

Results go to Drive, not local disk. Colab wipes local storage on
disconnect, and the whole resumability design depends on completed cells
outliving the session that produced them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE   = '/content/drive/MyDrive/mtg-bdh'
DATA    = f'{DRIVE}/data/FIN.PremierDraft'
RESULTS = f'{DRIVE}/runs/grid'

import os
assert os.path.exists(f'{DATA}/picks.npz'), f'upload the processed data to {DATA}'
os.makedirs(RESULTS, exist_ok=True)
print('data ok, results ->', RESULTS)

In [ ]:
REPO   = 'https://github.com/roro2006/MTG-bdh-architecture-and-scaling.git'
BRANCH = 'worktree-density-plumbing'

!git clone --branch {BRANCH} {REPO} /content/mtg 2>/dev/null || (cd /content/mtg && git pull)
%cd /content/mtg
!pip install -q -r requirements.txt

## 4. Sanity check

Runs the suite before spending GPU time. `test_kernels.py` matters most
here: this is the first time the Pallas kernels have ever lowered on real
accelerator hardware. It asserts `pallas_call` actually appears in the
jaxpr, so a silent fallback to the reference path cannot pass for success.

In [ ]:
!python -m pytest -q tests/test_grid.py tests/test_models.py
!python -m pytest -q tests/test_kernels.py    # first real-hardware run

## 5. Calibrate the estimate before trusting it

Every hour-estimate for this grid is a FLOP roofline, and a roofline
misprices anything bound by memory traffic or kernel launches rather than
arithmetic — which small cells are. So measure one real cell and back out
the *achieved* throughput instead of assuming a number.

In [ ]:
import time, jax.numpy as jnp
from src.data.card_features import CardFeatures
from src.data.dataset import PickData, split_by_draft
from src.training.grid import GridCell, run_cell, run_grid, full_grid, pilot_grid, estimate

table  = jnp.asarray(CardFeatures.load(f'{DATA}/card_features.npz').dense())
data   = PickData.load(DATA)
splits = split_by_draft(data, seed=0)

probe = GridCell('attention', 226, 0.02, 0, role='pilot')
t0 = time.time()
r  = run_cell(probe, data, table, splits, batch_size=512, epochs=0.5)
elapsed = time.time() - t0

achieved = r['flops_per_example'] * r['examples_seen'] / elapsed / 1e12
print(f"achieved ~{achieved:.1f} TFLOP/s")
for name, g in (('pilot', pilot_grid()), ('full', full_grid())):
    e = estimate(g, achieved)
    print(f"  {name:6s} {e['cells']:3d} cells  {e['hours']:6.1f} h total, "
          f"largest cell {e['largest_cell_hours']:.1f} h")

## 6. Pilot grid

Cheap, and its only job is to fail. If the loss curves here are not sane,
nothing downstream is worth running. Both arms should land well below the
pick-rate prior of 1.5662.

In [ ]:
pilot = run_grid(pilot_grid(), data, table, splits, f'{RESULTS}/pilot',
                 batch_size=512, epochs=1.0)
for r in pilot:
    print(f"{r['name']:28s} N={r['num_params']:>10,} D={r['train_rows']:>9,} "
          f"best={r['best_val_loss']:.4f}")

## 7. Full grid

Safe to interrupt. Completed cells are skipped on restart, so when the
runtime disconnects just re-run this cell in a fresh session. Cells are
ordered most-expensive-first, so a session that dies has done the costly
work rather than saved it for last.

In [ ]:
results = run_grid(full_grid(), data, table, splits, RESULTS,
                   batch_size=512, epochs=1.0)
print(f"{len(results)} cells complete")

## 8. Progress

Run this any time to see what is left. Deleting a result file is how you
ask for that cell to be run again.

In [ ]:
from src.training.grid import load_results

done = {r['name'] for r in load_results(RESULTS)}
todo = [c for c in full_grid() if c.name not in done]
print(f"{len(done)} done, {len(todo)} remaining")
if todo:
    print("next:", ", ".join(c.name for c in todo[:5]))